In [5]:
from azureml.core import Workspace, Datastore, Dataset

ws = Workspace.from_config()

# 1. Provide your storage details
storage_account_name = "cloudproject60306117"
# !!! PASTE YOUR ACTUAL KEY HERE !!!
storage_key = "AdxIWpu+RCT/r8Gr8rvVZxi25XMS/T0QnFugMQMzWXp6P7RHljEIpEZooyARZ3VJAEalcNRoGaDu+AStblz2Bw==" 
container_name = "curated"

# 2. Register the Datastore in Azure ML
try:
    datastore = Datastore.register_azure_blob_container(
        workspace=ws, 
        datastore_name='gold_data_store', 
        container_name=container_name,
        account_name=storage_account_name,
        account_key=storage_key
    )
    print("Datastore 'gold_data_store' registered successfully!")
except Exception as e:
    print(f"Registration failed: {e}")

# 3. Now point to the Parquet files and create the Dataset
datastore_path = [(datastore, 'taxi_features_v1/*.parquet')]
taxi_ds = Dataset.Tabular.from_parquet_files(path=datastore_path)

# 4. Register the Dataset (for Traceability Requirement II.3)
taxi_ds = taxi_ds.register(workspace=ws, 
                           name='taxi_gold_ds', 
                           description='Taxi Gold Features for Phase 2', 
                           create_new_version=True)

# 5. Load into Pandas
df = taxi_ds.to_pandas_dataframe()
print(f"Success! Data loaded with {len(df)} rows.")

Datastore 'gold_data_store' registered successfully!
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Success! Data loaded with 496153 rows.


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [6]:
# Create the dataset from the Parquet files in your curated container
datastore_path = [(datastore, 'taxi_features_v1/*.parquet')]
taxi_ds = Dataset.Tabular.from_parquet_files(path=datastore_path)

# Register the dataset so it appears in your Azure ML "Data" tab
taxi_ds = taxi_ds.register(workspace=ws, 
                           name='taxi_gold_ds', 
                           description='Versioned features for Phase 2', 
                           create_new_version=True)

# Convert to Pandas for training
df = taxi_ds.to_pandas_dataframe()
print(f"Data Loaded: {df.shape[0]} rows ready for training.")

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
Data Loaded: 496153 rows ready for training.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# A. Reproducible Split (80/20)
train, test = train_test_split(df, test_size=0.2, random_state=42)

# B. Baseline (Simple)
X_train_base = train[['trip_distance', 'passenger_count']]
y_train = train['fare_amount']
base_model = LinearRegression().fit(X_train_base, y_train)
base_rmse = np.sqrt(mean_squared_error(test['fare_amount'], base_model.predict(test[['trip_distance', 'passenger_count']])))

# C. Full-Feature (Your Hypothesis)
features = ['trip_distance', 'passenger_count', 'pickup_hour', 'day_of_week', 'trip_duration_mins']
rf_model = RandomForestRegressor(n_estimators=100, random_state=42).fit(train[features], y_train)
full_rmse = np.sqrt(mean_squared_error(test['fare_amount'], rf_model.predict(test[features])))

print(f"Baseline RMSE: ${base_rmse:.2f}")
print(f"Full-Feature RMSE: ${full_rmse:.2f}")

Baseline RMSE: $10.55
Full-Feature RMSE: $2.49


In [8]:
import joblib
from azureml.core.model import Model

# Save locally
joblib.dump(value=rf_model, filename='taxi_model.pkl')

# Register to Azure
model = Model.register(workspace=ws,
                       model_path='taxi_model.pkl',
                       model_name='taxi_fare_predictor',
                       tags={'Algorithm': 'RandomForest', 'Baseline_RMSE': str(round(base_rmse,2))},
                       datasets=[('training_data', taxi_ds)]) # LINK DATA TO MODEL

Registering model taxi_fare_predictor


In [12]:
import os

# Create the score.py content
score_script = """
import json
import joblib
import numpy as np
import pandas as pd
import os
from azureml.core.model import Model

def init():
    global model
    # Use the name you registered: 'taxi_fare_predictor'
    model_path = Model.get_model_path('taxi_fare_predictor')
    model = joblib.load(model_path)

def run(raw_data):
    try:
        data = json.loads(raw_data)['data']
        input_df = pd.DataFrame(data)
        prediction = model.predict(input_df)
        return prediction.tolist()
    except Exception as e:
        return str(e)
"""

# Physically write the file to the current directory
with open("score.py", "w") as f:
    f.write(score_script)

# Check if the file exists now
if os.path.exists("score.py"):
    print("Success: score.py is physically present on disk.")
else:
    print("Error: score.py still wasn't created. Check your folder permissions.")

Success: score.py is physically present on disk.


In [13]:
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice
from azureml.core import Environment, Workspace
from azureml.core.model import Model

# 1. Connect to Workspace
ws = Workspace.from_config()

# 2. Define the environment using your env.yml
myenv = Environment.from_conda_specification(name="taxi-env", file_path="env.yml")

# 3. Configure Inference 
# source_directory="." ensures Azure uploads score.py from your current folder
inference_config = InferenceConfig(
    entry_script="score.py", 
    environment=myenv,
    source_directory="." 
)

# 4. Hardware setup 
deployment_config = AciWebservice.deploy_configuration(cpu_cores=1, memory_gb=1)

# 5. Reference the model you registered earlier
model = Model(ws, 'taxi_fare_predictor')

print("Starting deployment... This usually takes 5-10 minutes.")

# 6. Execute Deployment
service = Model.deploy(
    workspace=ws, 
    name='taxi-fare-endpoint', 
    models=[model], 
    inference_config=inference_config, 
    deployment_config=deployment_config,
    overwrite=True
)

service.wait_for_deployment(show_output=True)

print(f"\nPhase 2 Complete! Your Endpoint URL: {service.scoring_uri}")

/tmp/ipykernel_3301/465758312.py:29: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(
Service deployment polling reached non-successful terminal state, current service state: Transitioning
Operation ID: 339c0e4b-f23a-4268-b781-7ee332a4f44d
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client 'd303fbde-2990-4aff-9de2-24202003a48d' with object id '4b098a39-3e

Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2026-04-09 16:02:35+00:00 Creating Container Registry if not exists..
2026-04-09 16:12:36+00:00 Registering the environment..
2026-04-09 16:12:37+00:00 Building image..
2026-04-09 16:20:53+00:00 Generating deployment configuration..
2026-04-09 16:20:57+00:00 Submitting deployment to compute.
Failed


WebserviceException: WebserviceException:
	Message: Service deployment polling reached non-successful terminal state, current service state: Transitioning
Operation ID: 339c0e4b-f23a-4268-b781-7ee332a4f44d
More information can be found using '.get_logs()'
Error:
{
  "code": "AuthorizationFailed",
  "statusCode": 403,
  "message": "ACI Service request failed. Reason: The client 'd303fbde-2990-4aff-9de2-24202003a48d' with object id '4b098a39-3e93-43ef-b016-914593d5005d' does not have authorization to perform action 'Microsoft.ContainerInstance/containerGroups/write' over scope '/subscriptions/0b475409-9d7c-4dff-a07b-084eff651874/resourceGroups/rg-60306117/providers/Microsoft.ContainerInstance/containerGroups/taxi-fare-endpoint-h-mPz21VfkGG6m5qiyQlMg' or the scope is invalid. If access was recently granted, please refresh your credentials.."
}
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "Service deployment polling reached non-successful terminal state, current

In [14]:
import json
import joblib
import pandas as pd

# Manually trigger the 'init' and 'run' logic from score.py
model_path = 'taxi_model.pkl' # The file you saved earlier
local_model = joblib.load(model_path)

# Test data
test_input = {"data": [{"trip_distance": 5.0, "passenger_count": 1, "pickup_hour": 14, "day_of_week": 2, "trip_duration_mins": 15.0}]}
df_test = pd.DataFrame(test_input["data"])

# Get prediction
prediction = local_model.predict(df_test)
print(f"Local Test Prediction: ${prediction[0]:.2f}")

Local Test Prediction: $17.30


In [15]:
import joblib
import pandas as pd

# 1. Load your trained model file
model_local = joblib.load('taxi_model.pkl')

# 2. Simulate an API call with new data
# (5 miles, 1 passenger, 2:00 PM, Wednesday, 15 min duration)
sample_data = pd.DataFrame([{
    "trip_distance": 5.0, 
    "passenger_count": 1, 
    "pickup_hour": 14, 
    "day_of_week": 2, 
    "trip_duration_mins": 15.0
}])

# 3. Get the prediction
prediction = model_local.predict(sample_data)
print(f"Prediction Success! Predicted Fare: ${prediction[0]:.2f}")

Prediction Success! Predicted Fare: $17.30
